In [189]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [190]:
train = pd.read_csv('train.csv')

In [191]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [192]:
train['Age']= train['Age'].fillna(train['Age'].mean())

In [193]:
train= train.drop(columns=['Cabin','Ticket','Name'],axis=1)

In [194]:
train = pd.get_dummies(train,columns=['Embarked','Sex'],drop_first=True,dtype=int)

In [195]:
train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Sex_male
0,1,0,3,22.0,1,0,7.2500,0,1,1
1,2,1,1,38.0,1,0,71.2833,0,0,0
2,3,1,3,26.0,0,0,7.9250,0,1,0
3,4,1,1,35.0,1,0,53.1000,0,1,0
4,5,0,3,35.0,0,0,8.0500,0,1,1


In [196]:
x= train.drop('Survived',axis=1)
y = train['Survived']

In [197]:
x_train , x_test, y_train , y_test = train_test_split(x,y,test_size=.15,random_state=42)

In [198]:
model = DecisionTreeClassifier(max_depth=5,min_samples_split=10,criterion='log_loss')

In [199]:
model.fit(x_train,y_train)

,criterion,'log_loss'
,splitter,'best'
,max_depth,5
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [200]:
y_pred = model.predict(x_test)

In [201]:
from sklearn.metrics import accuracy_score

In [202]:
accuracy_score(y_test,y_pred)

0.7761194029850746

### Random Search CV

In [203]:
from sklearn.model_selection import RandomizedSearchCV

In [204]:
param_dist = {
    "max_depth": [None, 3, 5, 7, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["gini", "entropy", "log_loss"],
    "max_features": [None, "sqrt", "log2"]
}

In [205]:
rscv_df= DecisionTreeClassifier()

In [206]:
random_search = RandomizedSearchCV(
    estimator=rscv_df,
    param_distributions= param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=42
)

In [207]:
random_search.fit(x_train,y_train)

,estimator,DecisionTreeClassifier()
,param_distributions,"{'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 3, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [208]:
print(random_search.best_params_)
print(random_search.best_score_)

{'min_samples_split': 20, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': 3, 'criterion': 'log_loss'}
0.8111362844196585


In [209]:
y_pred_rs = random_search.predict(x_test)

In [210]:
accuracy_score(y_test,y_pred_rs)

0.8134328358208955

In [211]:
result = pd.DataFrame()

In [212]:
result['PassengerId'] = test['PassengerId']

Test SEt preproccessing

In [213]:
test

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Sex_male
0,892,3,34.500000,0,0,7.8292,1,0,1
1,893,3,47.000000,1,0,7.0000,0,1,0
2,894,2,62.000000,0,0,9.6875,1,0,1
3,895,3,27.000000,0,0,8.6625,0,1,1
4,896,3,22.000000,1,1,12.2875,0,1,0
...,...,...,...,...,...,...,...,...,...
413,1305,3,29.699118,0,0,8.0500,0,1,1
414,1306,1,39.000000,0,0,108.9000,0,0,0
415,1307,3,38.500000,0,0,7.2500,0,1,1
416,1308,3,29.699118,0,0,8.0500,0,1,1


In [214]:
test_pred = random_search.predict(test)

In [215]:
result['Survived'] = test_pred

In [216]:
result

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [218]:
result.to_csv('titanic-submission.csv',index=False)